# 模型调用 — init_chat_model() 函数

`init_chat_model()` 是 LangChain 中最常用的函数之一，它让你用统一的方式连接 20 多种模型提供商，不需要记忆每个提供商的类名和参数差异。

In [2]:
from dotenv import load_dotenv
load_dotenv()

from langchain.chat_models import init_chat_model

print("init_chat_model 导入成功")

init_chat_model 导入成功


`init_chat_model()` 的完整函数签名：

```python
def init_chat_model(
    model: str | None,                    # 模型名称（provider:model 格式）
    *,
    model_provider: str | None = None,    # 单独的模型提供商
    configurable_fields: Any = None,      # 可运行时修改的字段
    config_prefix: str | None = None,     # 配置键前缀
    **kwargs,                             # 模型特定参数
) -> BaseChatModel:
    ...
```

> 注意 `*` 是 Python 语法，表示后面的参数必须用关键字传参，不能按位置传参。

参数说明：

| 参数 | 类型 | 说明 | 默认值 |
| --- | --- | --- | --- |
| model | str 或 None | 模型名，`provider:model` 格式。传 None 时用于创建可配置模型 | 无 |
| model_provider | str 或 None | 单独指定提供商。动态获取或 model 无法推断时使用 | None |
| configurable_fields | "any" 或 list 或 None | 可运行时修改的字段列表。None 表示固定模型 | None |
| config_prefix | str 或 None | 多模型场景下配置键的前缀，避免冲突 | None |
| **kwargs | dict | 传递给底层模型的参数 | 无 |

## model 参数

### provider:model 格式（推荐）

格式为 `提供商:模型名`，用冒号分隔。这是最推荐的方式，显式指定了模型提供商。

In [3]:
# provider:model 格式 —— 显式指定提供商和模型
model = init_chat_model("deepseek:deepseek-v4-flash")
# model = init_chat_model("anthropic:claude-sonnet-4-5-20250929")
# model = init_chat_model("ollama:llama3.2")
# model = init_chat_model("groq:llama-3.3-70b")

### 自动推断模型提供商

如果不指定提供商前缀，LangChain 会尝试从模型名自动推断：

In [4]:
# 自动推断提供商（基于模型名前缀）
model = init_chat_model("deepseek-v4-flash")    # → deepseek
# model = init_chat_model("claude-sonnet-4-5")     # → anthropic
# model = init_chat_model("gpt-4o")               # → openai
# model = init_chat_model("grok-3")               # → xai
# model = init_chat_model("mistral-large")        # → mistralai

print("取消注释以上行即可使用对应模型")

取消注释以上行即可使用对应模型


| 模型名前缀 | 推断提供商 |
| --- | --- |
| gpt-、o1、o3、chatgpt、text-davinci | openai |
| claude | anthropic |
| gemini | google_vertexai |
| command | cohere |
| deepseek | deepseek |
| mistral、mixtral | mistralai |
| grok | xai |
| sonar | perplexity |
| amazon.、anthropic.、meta. | bedrock |

> 自动推断虽然方便，但不保证 100% 正确。生产环境建议始终使用 `provider:model` 格式。

### model_provider 参数

当 `model_provider` 单独指定时，效果等价于 `provider:model` 格式。使用场景：
- 从配置文件动态读取提供商名称时
- 提供商名称和模型名需要独立配置时
- 模型名无法被自动推断且不方便拼接字符串时

In [5]:
# 下面两种写法完全等价
# model = init_chat_model("claude-sonnet-4-5", model_provider="anthropic")
# model = init_chat_model("anthropic:claude-sonnet-4-5")

print("两种写法等价：显式指定 provider")

两种写法等价：显式指定 provider


## 固定模型 vs 可配置模型

`init_chat_model()` 有两种使用模式。

### 模式 1：固定模型

指定具体的 model 字符串，返回可直接使用的 BaseChatModel 实例。

In [6]:
# 模式 1：固定模型 —— 返回可直接调用的模型实例
model = init_chat_model("deepseek:deepseek-v4-flash", temperature=0.7)

# 取消注释以运行
response = model.invoke("介绍一下你自己（一句话）")
print(response.content)

print(f"模型已创建: {model.__class__.__name__}")

你好！我是DeepSeek，由深度求索公司创造的文本型AI助手，专注于通过对话为你提供信息、解答问题和协助完成各种任务。
模型已创建: ChatDeepSeek


### 模式 2：可配置模型

不指定 model（或设为 None），创建可在运行时动态切换的模型。

In [8]:
# 模式 2：可配置模型 —— 运行时动态指定模型
configurable_model = init_chat_model(
    None,                  # 不固定模型
    temperature=0.7,       # 默认温度
    configurable_fields=["model"],  # 只允许运行时修改 model
)

# 运行时通过 config 指定模型
response = configurable_model.invoke(
    "介绍一下你自己（一句话）",
    config={"configurable": {"model": "deepseek:deepseek-v4-flash"}}
)
print(response.content)

print("可配置模型已创建，运行时通过 config 切换模型")

你好，我是DeepSeek，由深度求索公司创造的免费AI助手，擅长通过超长上下文（1M tokens）处理复杂任务，支持文件上传、联网搜索，并始终以热情细腻的方式为你提供帮助！
可配置模型已创建，运行时通过 config 切换模型


> 可配置模型在 A/B 测试和成本优化中非常有用。可以在不重启服务的情况下通过修改配置来切换模型。

`configurable_fields` 的取值：

| 值 | 含义 |
| --- | --- |
| None | 不可配置，返回普通的 BaseChatModel（固定模型模式） |
| "any" | 所有参数可配置（注意安全风险） |
| ["model", "temperature"] | 只有列表中指定的字段可配置 |

> 使用 `configurable_fields="any"` 时要注意安全：api_key 和 base_url 等敏感字段可能被篡改。生产环境建议显式列出可配置字段。

## 所有支持的模型提供商

以下是 `init_chat_model()` 内置支持的提供商及其安装包：

| provider 名称 | 安装包 | 代表模型 |
| --- | --- | --- |
| openai | langchain-openai | gpt-4o、gpt-4o-mini |
| anthropic | langchain-anthropic | claude-sonnet-4-5 |
| google_genai | langchain-google-genai | gemini-2.5-flash |
| google_vertexai | langchain-google-vertexai | gemini-2.5-pro |
| deepseek | langchain-deepseek | deepseek-v4-flash、deepseek-v4-pro |
| mistralai | langchain-mistralai | mistral-large |
| groq | langchain-groq | llama-3.3-70b |
| ollama | langchain-ollama | llama3.2、qwen2.5 |
| fireworks | langchain-fireworks | llama-v3p1-70b |
| together | langchain-together | Llama-3.3-70B |
| xai | langchain-xai | grok-3 |
| openrouter | langchain-openrouter | openai/gpt-4o |
| perplexity | langchain-perplexity | sonar-pro |
| huggingface | langchain-huggingface | 各类 HuggingFace 模型 |
| cohere | langchain-cohere | command-r-plus |

## 常用 kwargs 参数

kwargs 参数会直接传递给底层模型类，常用的包括：

In [9]:
model = init_chat_model(
    "deepseek:deepseek-v4-flash",
    temperature=0.3,     # 控制输出随机性（0~2），值越小输出越稳定
    max_tokens=200,      # 限制输出最大 token 数（控制成本）
    timeout=30,          # 请求超时时间（秒）
    max_retries=2,       # 失败重试次数
)

response = model.invoke("1+1=？")
print(response.content)

print(f"模型已创建: {model.__class__.__name__}")

1+1=2
模型已创建: ChatDeepSeek


| 参数 | 类型 | 说明 |
| --- | --- | --- |
| temperature | float | 控制随机性，0~2 |
| max_tokens | int | 限制输出最大 Token 数 |
| timeout | int/float | 请求超时秒数 |
| max_retries | int | 请求失败后的重试次数 |
| base_url | str | 自定义 API 端点（代理/中转） |
| rate_limiter | BaseRateLimiter | 速率限制器实例 |
| top_p | float | 核采样参数，0~1 |
| stop | list[str] | 停止序列 |

> `temperature` 和 `top_p` 通常不同时设置。对于大多数场景，只调整 `temperature` 就足够了。

## ConfigurableModel —— 高级用法

允许在运行时动态指定模型和参数，适合 A/B 测试与多模型切换。

In [ ]:
# 创建可配置模型，设置默认值
model = init_chat_model(
    "deepseek:deepseek-v4-flash",     # 默认模型
    configurable_fields="any",         # 所有参数都可在运行时修改
    config_prefix="my",                # 配置键前缀
    temperature=0.3,                    # 默认温度
)

# 使用默认配置运行（取消注释以执行）
response = model.invoke("介绍拜登（一句话）")
print(f"默认: {response.content[:50]}...")

# 运行时覆盖模型和参数（注意 my_ 前缀）
response = model.invoke(
    "介绍小布什（一句话）",
    config={
        "configurable": {
            "my_model": "deepseek:deepseek-v4-pro",   # 切换模型
            "my_temperature": 0.9,                    # 调整温度
        }
    }
)
print(f"覆盖: {response.content[:50]}...")

默认: 乔·拜登是美国第46任总统，曾任副总统和参议员，以其数十年政治经验和推动国内经济复苏、气候变化应对及...
覆盖: 乔治·沃克·布什（小布什）是美国第43任总统（2001-2009年），其任期因“9·11”恐怖袭击事...
可配置模型已创建，取消注释上方 invoke 调用即可测试


**术语：**

- **init_chat_model()**：LangChain 的统一模型初始化函数，自动根据模型标识选择对应的类
- **BaseChatModel**：所有聊天模型的基类，提供 invoke/stream 等统一接口
- **ConfigurableModel**：支持运行时动态切换模型的高级封装
- **provider:model 格式**：`deepseek:deepseek-v4-flash` 这种用冒号分隔的模型标识方式